In [ ]:
# Source tracing workflow
# This notebook traces source facilities and contributing chemicals for the study area
# by combining TOXCONC data from the RSEI water microdata (flowlines) to the RSEI public release tables.
# The results are written to the results folder as CSV files.

In [ ]:
import pandas as pd
import geopandas as gpd
from utils.config import root_dir
from utils.rsei_utils import (
    load_tabular_water_microdata,
    get_source_chemicals,
    get_source_facilities,
)

In [ ]:
%%time
# load the data
aoi = gpd.read_file(root_dir + "results/aoi_huc12_boundaries.gpkg")
years = range(2008, 2023)
universe = "core01"
# pre-load tabular Geographic Microdata
tabular_gm_on = load_tabular_water_microdata(site="Onsite", universe=universe)
tabular_gm_off = load_tabular_water_microdata(site="Offsite", universe=universe)

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


CPU times: user 1min 37s, sys: 3.96 s, total: 1min 41s
Wall time: 1min 41s


### Identify source facilities

In [ ]:
# get facilities from on- and off-site releases
facilities_on = get_source_facilities(
    "Onsite", tabular_gm_on, aoi=aoi, universe=universe, years=years
)
facilities_off = get_source_facilities(
    "Offsite", tabular_gm_off, aoi=aoi, universe=universe, years=years
)
cols = [
    "FacilityID",
    "FacilityName",
    "Latitude",
    "Longitude",
    "Street",
    "City",
    "County",
    "State",
    "ZIPCode",
    "2022NAICSCode",
    "TRIIndustrySector",
    "LongName",
]
facilities_on["site"] = "on"
facilities_off["site"] = "off"
# Concatenate
facilities_all = pd.concat([facilities_on, facilities_off])
# Define the aggregation logic
# Replace 'facility_id' with the actual column name you use to identify facilities
aggregation_rules = {
    "ToxConc": "sum",  # Sum the concentrations
    "site": lambda x: (
        "both" if x.nunique() > 1 else x.iloc[0]
    ),  # Set 'both' if in both dfs
}
# Group by facility and apply the rules
facilities_all = (
    facilities_all.groupby(cols)
    .agg(aggregation_rules)
    .reset_index()
    .sort_values(by="ToxConc", ascending=False)
)
facilities_all["share ToxConc"] = (
    facilities_all["ToxConc"] / facilities_all["ToxConc"].sum() * 100
)
print(len(facilities_all), facilities_all.head(10)["share ToxConc"].sum())
facilities_all.head(10)
facilities_all.to_csv(root_dir + "results/source_facilities.csv")

Got 673 unique facilities for the study period
Got 800 unique facilities for the study period
1278 79.45241903260327


### Identify contributing chemicals

In [ ]:
chemicals_on = get_source_chemicals(
    "Onsite", tabular_gm_on, aoi=aoi, universe=universe, years=years
)
chemicals_off = get_source_chemicals(
    "Offsite", tabular_gm_off, aoi=aoi, universe=universe, years=years
)
chemicals_all = pd.concat([chemicals_on, chemicals_off])
chemicals_all = chemicals_all.drop("ChemicalNumber", axis=1)
aggregation_rules = {"Chemical": lambda x: ", ".join(x.unique()), "ToxConc": "sum"}
# using MetalCombinedChemNum to combine metals and their associated compounds into one
chemicals_all = (
    chemicals_all.groupby(["MetalCombinedChemNum"]).agg(aggregation_rules).reset_index()
)
chemicals_all = chemicals_all.sort_values(by="ToxConc", ascending=False)
chemicals_all["share ToxConc"] = (
    chemicals_all["ToxConc"] / chemicals_all["ToxConc"].sum() * 100
)
print(len(chemicals_all), chemicals_all.head(10)["share ToxConc"].sum())
chemicals_all.head(10)
chemicals_all.to_csv(root_dir + "results/source_chemicals.csv")

Got 186 unique chemicals for the study period
Got 121 unique chemicals for the study period
193 93.39167030416277
